In [ ]:
# ======================================================================
# Seasonal LOCA2 Statistical 3-km Anomaly Time Series
#   - Regions: Joshua Tree NP, Mojave NP
#   - Variables: T_Avg (from T_Max + T_Min), Precip
#   - Scenarios: SSP 2-4.5, SSP 3-7.0, SSP 5-8.5
#   - Up to 2 simulations per scenario (averaged)
#   - Seasonal (DJF/MAM/JJA/SON) anomalies relative to 1995–2014
#   - Aggregated into 10-year bins (1950–1959, 1960–1969, ...)
#   - Output: CSV for later plotting in R
# ======================================================================

import climakitae as ck
from climakitae.core.data_interface import get_data

import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray  # needed for .rio.* methods
import warnings
import os
import gc

# -------------------------
# Global config
# -------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

xr.set_options(keep_attrs=True)

STANDARD_CRS   = "EPSG:4326"
DOWNSCALING    = "Statistical"
RESOLUTION     = "3 km"
TIMESCALE      = "monthly"

# Time span / baseline
FULL_TIMESPAN  = (1950, 2100)
BASELINE_YEARS = (1995, 2014)

# We will use only these SSPs
SCENARIOS = ["SSP 2-4.5", "SSP 3-7.0", "SSP 5-8.5"]

# LOCA2 Statistical variable names
VARIABLES_STAT = {
    "T_Max":  "Maximum air temperature at 2m",
    "T_Min":  "Minimum air temperature at 2m",
    "Precip": "Precipitation (total)",
}

# Regions & shapefiles (update paths as needed)
SHAPEFILES = {
    "JoshuaTree": "../JoshuaTreeOutlines/JoshuaTree/Joshua_Tree_National_Park.shp",
    "Mojave":     "../Mojave/Mojave_National_Preserve.shp",
}

# Output
output_folder = "dataForRScripts"
os.makedirs(output_folder, exist_ok=True)
output_csv = os.path.join(
    output_folder,
    "Statistical_3km_Seasonal_Tavg_Precip_Decadal_Anomalies.csv"
)

# ======================================================================
# Helper functions
# ======================================================================

def load_boundary(name, path):
    """Load shapefile into WGS84 GeoDataFrame."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Shapefile for {name} not found at: {path}")
    gdf = gpd.read_file(path)
    if gdf.crs is None or str(gdf.crs) != STANDARD_CRS:
        gdf = gdf.to_crs(STANDARD_CRS)
    return gdf


def identify_spatial_dims(da):
    """Determine x and y dimension names for LOCA2 Statistical data."""
    y_dim = None
    x_dim = None

    # y / latitude
    for cand in ["lat", "latitude", "y"]:
        if cand in da.dims:
            y_dim = cand
            break

    # x / longitude
    for cand in ["lon", "longitude", "x"]:
        if cand in da.dims:
            x_dim = cand
            break

    if x_dim is None or y_dim is None:
        raise ValueError(f"Could not identify spatial dims in {da.dims}")

    return x_dim, y_dim


def ensure_crs_and_spatial_dims(da):
    """
    Ensure rioxarray has CRS and knows which dims are x/y.
    Returns a DataArray ready for clipping and area-weighted averaging.
    """
    da = da.copy()
    x_dim, y_dim = identify_spatial_dims(da)

    # CRS
    if da.rio.crs is None:
        da = da.rio.write_crs(STANDARD_CRS)

    # Spatial dims
    da = da.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim)

    # Drop any 2D lat/lon coords that might conflict
    for coord in ["lat", "lon", "latitude", "longitude"]:
        if coord in da.coords and da[coord].ndim == 2:
            da = da.drop_vars(coord, errors="ignore")

    return da, x_dim, y_dim


def clip_and_area_average(da, boundary_gdf):
    """
    Clip DataArray to boundary and compute area-weighted spatial average.
    Returns DataArray with dims (time, simulation).
    """
    da = da.copy()

    # Ensure CRS & dims
    da, x_dim, y_dim = ensure_crs_and_spatial_dims(da)

    # Match boundary CRS → raster CRS (should both be WGS84, but just in case)
    if str(boundary_gdf.crs) != str(da.rio.crs):
        boundary_native = boundary_gdf.to_crs(da.rio.crs)
    else:
        boundary_native = boundary_gdf

    # Clip
    clipped = da.rio.clip(
        boundary_native.geometry.values,
        all_touched=True,
        drop=False
    )

    # Compute area-weighted average over space
    lat = clipped[y_dim]
    weights = np.cos(np.deg2rad(lat))
    weights.name = "weights"

    try:
        spatial_avg = clipped.weighted(weights).mean(dim=[x_dim, y_dim], skipna=True)
    except Exception as e:
        print(f"    Weighted mean failed ({e}); using simple mean.")
        spatial_avg = clipped.mean(dim=[x_dim, y_dim], skipna=True)

    spatial_avg.load()
    return spatial_avg


def convert_units(da, var_key):
    """
    Convert units:
      - T_* from K → °C (if needed)
      - Precip from kg m-2 s-1 → mm/month (if needed)
    """
    da = da.copy()

    # Temperature
    if var_key in ["T_Max", "T_Min"]:
        units = da.attrs.get("units", "").lower()
        if units in ["k", "kelvin"]:
            da = da - 273.15
            da.attrs["units"] = "°C"

    # Precipitation
    if var_key == "Precip":
        units = da.attrs.get("units", "").replace(" ", "")
        # typical: "kgm-2s-1" or "kg/m^2/s"
        if "kgm-2s-1" in units or "kg/m^2/s" in units:
            # Convert flux to accumulation per month: [kg m-2 s-1] ≈ [mm s-1]
            days_in_month = da.time.dt.days_in_month
            seconds_per_day = 86400
            da = da * seconds_per_day * days_in_month
            da.attrs["units"] = "mm/month"
        # If it's already mm/month, we'll just trust it.

    return da


def unify_simulation_dim(da):
    """
    Ensure we have a 'simulation' dimension.
    - If 'simulation' exists, keep it.
    - If 'source_id' exists, rename to 'simulation'.
    - If neither exists, create a singleton 'simulation' dim.
    """
    da = da.copy()
    sim_dim = None

    if "simulation" in da.dims:
        sim_dim = "simulation"
    elif "source_id" in da.dims:
        da = da.rename({"source_id": "simulation"})
        sim_dim = "simulation"

    if sim_dim is None:
        # Add a singleton dimension called 'simulation'
        da = da.expand_dims({"simulation": [0]})
        sim_dim = "simulation"

    return da, sim_dim


def limit_to_two_simulations(da, sim_dim):
    """Slice to at most two simulations."""
    n_sims = da.sizes.get(sim_dim, 1)
    n_keep = min(2, n_sims)
    return da.isel({sim_dim: slice(0, n_keep)})


def fetch_monthly_field(region_name, boundary_gdf, scenario, var_key):
    """
    For a single region + scenario + variable:
      - Retrieve LOCA2 Statistical 3-km monthly data 1950–2100
      - Limit to bounding box around region
      - Convert units
      - Clip and area-average
      - Return DataArray with dims (time, simulation)
    """
    var_name = VARIABLES_STAT[var_key]

    # Use small bounding box around region to keep data size manageable
    minx, miny, maxx, maxy = boundary_gdf.total_bounds
    buffer = 0.1  # degrees; small margin
    lon_slice = (minx - buffer, maxx + buffer)
    lat_slice = (miny - buffer, maxy + buffer)

    print(f"    Fetching {var_key} ({var_name}) for {region_name}, {scenario}...")
    ds = get_data(
        variable=var_name,
        resolution=RESOLUTION,
        downscaling_method=DOWNSCALING,
        timescale=TIMESCALE,
        scenario=[scenario],
        time_slice=FULL_TIMESPAN,
        latitude=lat_slice,
        longitude=lon_slice
    )

    if ds is None:
        print(f"      Warning: get_data returned None for {var_name}, {scenario}.")
        return None

    # If Dataset, grab the first data_var
    if isinstance(ds, xr.Dataset):
        data_vars = list(ds.data_vars)
        if not data_vars:
            print(f"      Dataset empty for {var_name}, {scenario}.")
            return None
        da = ds[data_vars[0]]
    else:
        da = ds

    # If there's a 'scenario' dim/coord with length 1, drop it
    if "scenario" in da.dims and da.sizes["scenario"] == 1:
        da = da.isel(scenario=0, drop=True)
    elif "scenario" in da.coords and len(da["scenario"]) == 1:
        da = da.drop_vars("scenario")

    # Ensure time is a proper dimension
    if "time" not in da.dims:
        raise ValueError("Expected 'time' dimension in LOCA2 output.")

    # Unit conversions
    da = convert_units(da, var_key)

    # Simulation dim
    da, sim_dim = unify_simulation_dim(da)
    da = limit_to_two_simulations(da, sim_dim)

    # Clip and area-average
    spatial_avg = clip_and_area_average(da, boundary_gdf)  # (time, simulation)

    # Add metadata
    spatial_avg.attrs["scenario_label"] = scenario
    spatial_avg.attrs["region_name"] = region_name
    spatial_avg.attrs["variable_key"] = var_key

    return spatial_avg


def compute_seasonal_decadal_anomalies(spatial_avg, var_key, region_name, scenario_label):
    """
    Given monthly spatial averages (time, simulation) for one var+region+scenario:
    - Compute seasonal climatology baseline (1995–2014) per simulation & season
    - Compute anomalies (Δ°C or % change) for each month
    - Bin into 10-year periods and average anomalies over:
        - all months in the bin
        - all simulations (up to 2)
    Returns a DataFrame with columns:
        Region, Scenario, Variable, Season, Year, Anomaly
    """
    if spatial_avg is None or spatial_avg.time.size == 0:
        return None

    # Convert to DataFrame
    df = spatial_avg.to_dataframe(name="Value").reset_index()
    # columns now include: 'time', 'simulation', 'Value' (plus any coords)

    # Ensure datetime
    df["Year"] = df["time"].dt.year

    # --- Build seasons manually from month ---
    month = df["time"].dt.month
    # Climatological seasons:
    # DJF = Dec, Jan, Feb
    # MAM = Mar, Apr, May
    # JJA = Jun, Jul, Aug
    # SON = Sep, Oct, Nov
    df["Season"] = np.select(
        [
            month.isin([12, 1, 2]),
            month.isin([3, 4, 5]),
            month.isin([6, 7, 8]),
            month.isin([9, 10, 11]),
        ],
        ["DJF", "MAM", "JJA", "SON"],
        default="UNKNOWN",
    )

    # Create a "Simulation" column
    if "simulation" in df.columns:
        df = df.rename(columns={"simulation": "Simulation"})
    else:
        df["Simulation"] = 0

    # --- Baseline seasonal mean per Simulation × Season ---
    base_mask = (df["Year"] >= BASELINE_YEARS[0]) & (df["Year"] <= BASELINE_YEARS[1])
    df_base = df[base_mask].copy()

    if df_base.empty:
        print(f"      Warning: no baseline data for {scenario_label} in {region_name}.")
        return None

    baseline = (
        df_base.groupby(["Simulation", "Season"])["Value"]
        .mean()
        .reset_index()
        .rename(columns={"Value": "Baseline"})
    )

    # Merge baseline back
    df = df.merge(baseline, on=["Simulation", "Season"], how="left")

    # --- Compute anomalies ---
    if var_key == "Precip":
        # baseline is in mm/month (on average for that season)
        threshold = 1.0  # mm/month; avoid insane % for near-zero baseline

        def precip_anom(row):
            b = row["Baseline"]
            v = row["Value"]
            if pd.isna(b) or abs(b) < threshold:
                return np.nan
            return (v - b) / b * 100.0

        df["Anomaly"] = df.apply(precip_anom, axis=1)
    else:
        # Temperature: absolute change (same units as Value)
        df["Anomaly"] = df["Value"] - df["Baseline"]

    # --- 10-year bins ---
    df["DecadeStart"] = (df["Year"] // 10) * 10
    df["YearCenter"] = df["DecadeStart"] + 5

    # --- Aggregate over time & simulation ---
    df["Region"]   = region_name
    df["Scenario"] = scenario_label
    df["Variable"] = "T_Avg" if var_key == "T_Avg" else var_key

    grouped = (
        df.groupby(["Region", "Scenario", "Variable", "Season", "YearCenter"])["Anomaly"]
        .mean()  # <-- no skipna arg; NaNs ignored by default
        .reset_index()
        .rename(columns={"YearCenter": "Year"})
    )

    # Drop rows with all-NaN anomalies (e.g., ultra-dry baseline seasons)
    grouped = grouped.dropna(subset=["Anomaly"], how="all")

    if grouped.empty:
        return None

    return grouped


# ======================================================================
# Main processing
# ======================================================================

all_results = []

for region_name, shp_path in SHAPEFILES.items():
    print(f"\n=== Region: {region_name} ===")
    try:
        boundary = load_boundary(region_name, shp_path)
    except Exception as e:
        print(f"  ERROR loading boundary for {region_name}: {e}")
        continue

    for scenario in SCENARIOS:
        print(f"\n  --- Scenario: {scenario} ---")

        data_dict = {}

        # 1. Fetch T_Max, T_Min, Precip monthly spatial averages
        for var_key in ["T_Max", "T_Min", "Precip"]:
            try:
                data_dict[var_key] = fetch_monthly_field(
                    region_name, boundary, scenario, var_key
                )
            except Exception as e:
                print(f"    ERROR fetching {var_key} for {scenario}: {e}")
                data_dict[var_key] = None

        # 2. Compute T_Avg = (T_Max + T_Min)/2 if both available
        if data_dict["T_Max"] is not None and data_dict["T_Min"] is not None:
            print("    Computing T_Avg from T_Max and T_Min...")
            # Align in case of small differences (time or simulation dims)
            tmax, tmin = xr.align(data_dict["T_Max"], data_dict["T_Min"], join="inner")
            tavg = (tmax + tmin) / 2.0
            tavg.attrs["units"] = tmax.attrs.get("units", "°C")
            data_dict["T_Avg"] = tavg
        else:
            print("    Missing T_Max or T_Min; skipping T_Avg for this scenario.")
            data_dict["T_Avg"] = None

        # 3. Compute seasonal decadal anomalies for T_Avg and Precip
        for var_key in ["T_Avg", "Precip"]:
            da = data_dict.get(var_key)
            if da is None:
                continue

            print(f"    Computing seasonal decadal anomalies for {var_key}...")
            result_df = compute_seasonal_decadal_anomalies(
                da, var_key, region_name, scenario
            )

            if result_df is not None and not result_df.empty:
                all_results.append(result_df)

        # Cleanup
        del data_dict
        gc.collect()

# ======================================================================
# Save combined CSV
# ======================================================================

if all_results:
    final_df = pd.concat(all_results, ignore_index=True)

    # Optional: sort for nice ordering
    final_df = final_df.sort_values(
        by=["Region", "Variable", "Season", "Scenario", "Year"]
    )

    final_df.to_csv(output_csv, index=False)
    print("\n===============================================")
    print(f"Finished. Wrote seasonal decadal anomalies to:\n  {output_csv}")
    print("Columns:", list(final_df.columns))
else:
    print("\nNo results were generated; CSV not written.")


=== Region: JoshuaTree ===

  --- Scenario: SSP 2-4.5 ---
    Fetching T_Max (Maximum air temperature at 2m) for JoshuaTree, SSP 2-4.5...
    Fetching T_Min (Minimum air temperature at 2m) for JoshuaTree, SSP 2-4.5...
    Fetching Precip (Precipitation (total)) for JoshuaTree, SSP 2-4.5...
    Computing T_Avg from T_Max and T_Min...
    Computing seasonal decadal anomalies for T_Avg...
    Computing seasonal decadal anomalies for Precip...

  --- Scenario: SSP 3-7.0 ---
    Fetching T_Max (Maximum air temperature at 2m) for JoshuaTree, SSP 3-7.0...
    Fetching T_Min (Minimum air temperature at 2m) for JoshuaTree, SSP 3-7.0...
    Fetching Precip (Precipitation (total)) for JoshuaTree, SSP 3-7.0...
    Computing T_Avg from T_Max and T_Min...
    Computing seasonal decadal anomalies for T_Avg...
    Computing seasonal decadal anomalies for Precip...

  --- Scenario: SSP 5-8.5 ---
    Fetching T_Max (Maximum air temperature at 2m) for JoshuaTree, SSP 5-8.5...
    Fetching T_Min (Minimum